# 03. 개선 실험 (Mission 2 · 3)

## 📐 Mission 2. 개선 전략 설계

### 1. 출발점: Baseline의 문제점 (Mission 1)
| 번호 | 문제점 | 근거 |
|---|---|---|
| P1 | 학습·테스트 분포 차이 → 검증 성능이 실제 성능을 대변하지 못함 | 학습 224×224 가공(여백 포함, 밝기 88.7) vs 테스트 원본 크기(여백 없음, 밝기 121.0), 폐렴 확률 중앙값 검증 0.979 → 테스트 0.503 |
| P2 | 지름길 학습 위험 | 검은 영역 비율 정상 17.6% vs 폐렴 31.5% |
| P3 | 클래스 불균형 | 폐렴 2.89배, 테스트는 정상 비율이 더 높음 |
| P4 | 모델 용량·해상도·학습 부족 | Conv 2층(약 7만 파라미터), 128×128 축소, 2 epochs에서 AUC 상승 중 |

### 2. 사전학습 모델 선정: ResNet18 (ImageNet)
- **데이터 규모에 맞는 크기**: 학습 약 4천 장 → 큰 모델(ResNet50, ViT)은 과적합 위험. ResNet18(약 1,100만 파라미터)은 적당한 용량
- **입력 크기 일치**: ImageNet 표준 입력 224×224 = 학습 이미지 크기
- **잔차 연결(residual connection)**: 층이 깊어져도 학습 신호가 약해지지 않아 안정적으로 학습
- **Grad-CAM 적용 용이**: 공식 라이브러리 권장 타겟 레이어 `layer4[-1]`
- **실험 속도**: T4 기준 에폭당 1분 안팎 → 여러 실험 반복 가능
- 흑백 1채널 → 3채널 복제, ImageNet 평균·표준편차로 정규화
- (비교 후보) DenseNet121: 흉부 X-ray 연구(CheXNet)에서 사용된 모델 → 시간 여유 시 추가 비교

### 3. Fine-tuning 방식: 직접 비교하여 결정
| 실험 | 고정(Freeze) | 학습 | 학습률 | 의도 |
|---|---|---|---|---|
| E1a 특징 추출 | 사전학습 층 전체 | 마지막 분류층(fc)만 | 1e-3 | ImageNet 특징을 그대로 쓰면 X-ray에서도 통하는가? |
| E1b 전체 미세조정 | 없음 | 모든 층 | 1e-4 (작게) | 사전학습 특징을 X-ray에 맞게 조정하면 더 나은가? |

- 예상: ImageNet(컬러 일반 사진)과 X-ray(흑백 의료 영상)는 성격이 크게 달라 **전체 미세조정이 유리**할 것
- 전체 미세조정은 학습률을 작게 하여 사전학습 지식을 급격히 망가뜨리지 않도록 함

### 4. 개선 전략 (모델 변경 외)
| 전략 | 해결하려는 문제 | 근거 | 영역 |
|---|---|---|---|
| **S1. 검은 여백 Crop** | P1, P2 | 학습 이미지에만 있는 검은 여백 제거 → 지름길 단서 차단 + 테스트 이미지 형태와 통일 | 데이터 |
| **S2a. CLAHE** | P1 (밝기) | 이미지마다 대비를 같은 기준으로 맞춤 → 테스트 전체가 밝은 "한 방향 치우침"에 직접 대응 | 데이터 |
| **S2b. 밝기·대비 증강** | P1 (밝기) | 학습 때 밝기를 무작위로 바꿔 다양한 밝기에 익숙해지게 함 | 데이터 |
| **S3. 클래스 가중치 손실** | P3 | 소수 클래스(정상) 오류에 더 큰 벌점 | 학습 방식 |
| S4. 학습률·에폭·스케줄러 | P4 | 2 epochs에서도 AUC 상승 중 | 학습 방식 |

- Crop · CLAHE는 **이미지 한 장 안에서만 계산**하므로 테스트 이미지에도 동일하게 적용 (테스트 통계를 사용하지 않음 → 규칙 준수)
- 증강은 학습 데이터에만 적용

### 5. 단계별 실험 설계: 한 번에 하나씩 바꿔 효과를 분리
| 단계 | 실험 | 바꾸는 것 | 답하려는 질문 | 비교 |
|---|---|---|---|---|
| 0 | E0 Baseline | - | 출발점 | - |
| 1 | E1a / E1b | 모델(ResNet18) + 224 해상도 + Fine-tuning 방식 | 사전학습 모델이 효과적인가? 어떤 Fine-tuning이 맞는가? | E0 vs E1a vs E1b |
| 2 | E2 | + Crop (S1) | 여백 제거가 테스트 성능과 검증-테스트 격차를 줄이는가? | E1(best) vs E2 |
| 2.5 | 밝기 재측정 | (학습 없음) | Crop 후에도 학습·테스트 밝기 차이가 남아 있는가? | Crop 전 vs 후 |
| 3 | E3a / E3b | + CLAHE (S2a) / + 증강 (S2b) | 밝기 문제엔 어떤 접근이 더 효과적인가? | E2 vs E3a vs E3b |
| 4 | E4 | + 클래스 가중치 손실 (S3) | 불균형 보정이 민감도·특이도 균형을 개선하는가? | E3(best) vs E4 |
| 5 | E5 | 학습률·에폭·스케줄러 (S4) | 학습 설정 조정으로 추가 향상이 있는가? | E4 vs E5 |

**2.5단계 판단 기준**
- Crop 후 밝기 차이가 거의 없으면 → 밝기 차이의 대부분은 검은 여백 때문이었음 → S2는 가볍게(증강 위주)
- 차이가 남아 있으면 → 실제 밝기 차이 존재 → CLAHE와 증강을 본격 비교
- 어느 쪽이든 E3a / E3b를 모두 실행하여 결과로 확인

### 6. 제외한 전략과 이유
| 제외 | 이유 |
|---|---|
| 외부 데이터 · 원본 데이터 추가 | 대회 규칙 위반 |
| 테스트 데이터로 정규화 · 의사 라벨링 | 테스트 데이터 학습 사용 금지 |
| 테스트 클래스 비율에 맞춘 임계값 조정 | 테스트 정보를 설정에 사용 → 임계값은 검증셋으로만 결정 |
| 좌우 반전 증강 | 심장 위치(왼쪽)가 뒤바뀐, 해부학적으로 존재하지 않는 이미지 생성 |
| 회전 · 이동 증강 | 회전 시 생기는 검은 모서리가 Crop으로 없앤 여백 단서를 다시 만들 수 있음 |
| 오버샘플링 | 가중치 손실과 같은 문제(P3)를 다루므로 하나만 선택하여 비교를 단순화 |
| 큰 모델(ResNet50, ViT 등) | 데이터 규모 대비 과적합 위험, 실험 시간 부담 |
| 앙상블 · TTA | 효과는 기대되나 개별 전략의 효과 분석이 흐려짐 → 시간 여유 시 마지막 단계에서 검토 |

### 7. 실험 운영 원칙
- 모든 실험은 **동일 분할**(`split_train.csv`, `split_valid.csv`)과 **동일 Seed(42)** 사용
- E1부터 **5 epochs**, 에폭마다 검증 AUC를 확인하여 **가장 좋은 에폭의 모델** 사용 (검증셋은 평가에만 사용, 학습에는 미사용)
  - Baseline(2 epochs)과 에폭 수가 다르다는 점은 E1 비교 시 함께 고려
- 실험 결과는 드라이브 `results.csv`에 자동 누적, 제출 파일·모델도 실험 이름으로 저장
- **검증 점수와 리더보드 점수를 함께 기록** → 검증-테스트 격차가 줄어드는지도 개선 지표로 사용
- 리더보드 제출(1일 10회)은 검증 결과가 의미 있게 달라진 실험 위주로

In [ ]:
# Setup & reproducibility


import os
import random
import numpy as np
import torch

SEED = 42

def seed_everything(seed):
    # 파이썬 · numpy · 파이토치(CPU/GPU) 무작위 고정
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # GPU 연산을 매번 같은 방식으로
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# → 무작위 요소를 SEED=42로 고정하고 사용할 장치(GPU)를 확인하겠다

device: cuda
GPU: Tesla T4


In [ ]:
# Mount Drive & unzip data


import os
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/dacon_chest_xray"
for sub in ["data", "models", "submissions"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

ZIP_PATH = os.path.join(PROJECT_DIR, "data", "open.zip")
DATA_DIR = "/content/data"

if os.path.exists(ZIP_PATH):
    !unzip -q -o "{ZIP_PATH}" -d "{DATA_DIR}"
    print("unzip done:", os.listdir(DATA_DIR))
else:
    print("zip not found → 드라이브 data 폴더를 확인하세요")


# → 드라이브를 연결하고 데이터를 Colab 로컬(/content/data)에 풀겠다

Mounted at /content/drive
unzip done: ['train', 'sample_submission.csv', 'test', 'test.csv', 'train.csv']


In [ ]:
# Load fixed split & paths


import pandas as pd

# 02_Baseline에서 저장한 고정 분할 불러오기 → Baseline과 똑같은 학습·검증 데이터 사용
tr_df = pd.read_csv(os.path.join(PROJECT_DIR, "data", "split_train.csv"))
val_df = pd.read_csv(os.path.join(PROJECT_DIR, "data", "split_valid.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
SUB_DIR = os.path.join(PROJECT_DIR, "submissions")
# RESULTS_PATH : 모든 실험 결과를 한 줄씩 쌓아둘 기록표
RESULTS_PATH = os.path.join(PROJECT_DIR, "results.csv")

print(f"train {len(tr_df)} | valid {len(val_df)} | test {len(test_df)}")
print("train label ratio:", tr_df["label"].value_counts(normalize=True).sort_index().round(3).tolist())


# → Baseline과 동일한 고정 분할(train 4172 / valid 1044)을 불러오고 경로를 정리하겠다

train 4172 | valid 1044 | test 624
train label ratio: [0.257, 0.743]


In [ ]:
# Preprocessing (Crop / CLAHE) & Dataset


from PIL import Image
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG_SIZE = 224
# ImageNet 사전학습 모델이 학습할 때 쓴 RGB 평균·표준편차 → 같은 기준으로 정규화해야 사전학습 지식이 잘 맞음
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
# 이 밝기 이하를 "검은색"으로 봄 (EDA와 같은 기준)
BLACK_THRESHOLD = 5

def crop_black_border(img):
    """S1. 가로·세로 줄 전체가 검은 행·열(여백)을 잘라냄"""
    arr = np.array(img)
    # 행(가로줄)마다 가장 밝은 값이 기준보다 크면 = 내용이 있는 줄
    rows = np.where(arr.max(axis=1) > BLACK_THRESHOLD)[0]
    cols = np.where(arr.max(axis=0) > BLACK_THRESHOLD)[0]
    # 내용이 거의 없는 비정상 이미지는 그대로 반환 (안전장치)
    if len(rows) < 10 or len(cols) < 10:
        return img
    # 내용이 있는 첫 줄 ~ 마지막 줄까지만 남김
    return img.crop((cols[0], rows[0], cols[-1] + 1, rows[-1] + 1))

# clipLimit=2.0 : 대비를 키우는 정도의 상한 (너무 크면 잡음까지 강조)
# tileGridSize=(8, 8) : 이미지를 8×8 구역으로 나눠 구역마다 밝기 분포를 고르게 폄
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def apply_clahe(img):
    """S2a. 이미지마다 대비를 같은 기준으로 맞춤"""
    return Image.fromarray(CLAHE.apply(np.array(img)))

def preprocess(img, cfg):
    # 순서 : Crop 먼저 → CLAHE (검은 여백이 CLAHE 계산에 섞이지 않도록)
    if cfg["crop"]:
        img = crop_black_border(img)
    if cfg["clahe"]:
        img = apply_clahe(img)
    return img

def build_transform(train, cfg):
    t = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    # S2b. 밝기·대비 증강은 학습 데이터에만 적용
    # brightness=0.3 : 밝기를 0.7~1.3배 사이에서 무작위로 / contrast=0.3 : 대비도 같은 범위
    if train and cfg["aug"]:
        t.append(transforms.ColorJitter(brightness=0.3, contrast=0.3))
    t += [
        # 흑백 1채널을 3채널로 복제 (사전학습 모델은 RGB 3채널 입력)
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
    return transforms.Compose(t)

class XrayDataset(Dataset):
    def __init__(self, df, root, cfg, train, has_label=True):
        self.files = df["file_name"].tolist()
        self.labels = df["label"].astype(int).tolist() if has_label else None
        self.root, self.cfg, self.has_label = root, cfg, has_label
        self.tf = build_transform(train, cfg)
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.files[i])).convert("L")
        x = self.tf(preprocess(img, self.cfg))
        return (x, self.labels[i]) if self.has_label else x

# DataLoader 작업자(worker)마다 Seed 고정 → 여러 프로세스로 읽어도 재현 가능
def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def make_loaders(cfg):
    g = torch.Generator().manual_seed(SEED)
    # num_workers=2 : 이미지 읽기를 2개 프로세스로 병렬 처리 → 속도 향상
    kw = dict(num_workers=2, worker_init_fn=seed_worker)
    train_loader = DataLoader(XrayDataset(tr_df, TRAIN_IMG_DIR, cfg, train=True),
                              batch_size=32, shuffle=True, generator=g, **kw)
    val_loader = DataLoader(XrayDataset(val_df, TRAIN_IMG_DIR, cfg, train=False),
                            batch_size=64, shuffle=False, **kw)
    test_loader = DataLoader(XrayDataset(test_df, TEST_IMG_DIR, cfg, train=False, has_label=False),
                             batch_size=64, shuffle=False, **kw)
    return train_loader, val_loader, test_loader


# → Crop·CLAHE·증강을 설정(cfg)으로 켜고 끌 수 있는 전처리와 Dataset을 만들어,
#   실험마다 설정만 바꾸면 같은 파이프라인으로 돌아가게 하겠다

In [ ]:
# Model builder (ResNet18, freeze options)


import torch.nn as nn
from torchvision import models

def build_model(freeze):
    # weights=DEFAULT : ImageNet으로 사전학습된 가중치 불러오기
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if freeze == "fc":
        # 특징 추출 : 사전학습 층의 가중치를 고정 (requires_grad=False → 업데이트 안 함)
        for p in m.parameters():
            p.requires_grad = False
    # 마지막 분류층 교체 : ImageNet 1000개 클래스 → 정상/폐렴 2개
    # 새로 만든 층은 기본적으로 requires_grad=True → 항상 학습됨
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device)

for mode in ["fc", "full"]:
    m = build_model(mode)
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in m.parameters())
    print(f"freeze={mode:4s} | trainable {n_train:,} / total {n_all:,}")
del m


# → ImageNet 사전학습 ResNet18의 분류층을 2클래스로 바꾸고,
#   "분류층만 학습" / "전체 학습" 두 방식에서 학습되는 가중치 수를 비교하겠다

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 95.0MB/s]


freeze=fc   | trainable 1,026 / total 11,177,538
freeze=full | trainable 11,177,538 / total 11,177,538


In [ ]:
# Train / evaluate / predict / experiment runner


import time, copy
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, f1_score

def evaluate(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.softmax(model(x.to(device)), dim=1)[:, 1]
            probs.extend(p.cpu().numpy()); labels.extend(y.numpy())
    probs, labels = np.array(probs), np.array(labels)
    preds = (probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    return {"acc": accuracy_score(labels, preds), "auc": roc_auc_score(labels, probs),
            "sens": tp / (tp + fn), "spec": tn / (tn + fp), "f1": f1_score(labels, preds),
            "fn": int(fn), "fp": int(fp), "probs": probs, "labels": labels}

def predict(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for x in loader:
            probs.extend(torch.softmax(model(x.to(device)), dim=1)[:, 1].cpu().numpy())
    return np.array(probs)

def log_result(row):
    # 기록표가 없으면 Baseline(E0) 결과로 시작 (02_Baseline 결과값)
    if os.path.exists(RESULTS_PATH):
        res = pd.read_csv(RESULTS_PATH)
    else:
        res = pd.DataFrame([{"exp": "E0", "desc": "Baseline SimpleCNN 128", "val_acc": 0.9454,
                             "val_auc": 0.9832, "val_sens": 0.9433, "val_spec": 0.9515, "val_f1": 0.9625,
                             "val_fn": 44, "val_fp": 13, "test_normal_ratio": 0.497, "test_prob_median": 0.503}])
    # 같은 실험 이름이 이미 있으면 지우고 새 결과로 교체 (다시 돌렸을 때 중복 방지)
    res = res[res["exp"] != row["exp"]]
    res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)
    res.to_csv(RESULTS_PATH, index=False)
    return res

def run_experiment(name, desc, cfg):
    t0 = time.time()
    # 실험마다 Seed 다시 고정 → 실험 순서와 관계없이 같은 결과
    seed_everything(SEED)
    train_loader, val_loader, test_loader = make_loaders(cfg)
    model = build_model(cfg["freeze"])

    # S3. 클래스 가중치 : 개수가 적은 클래스일수록 큰 가중치 (전체 / (2 × 클래스 개수))
    if cfg["weighted"]:
        counts = tr_df["label"].value_counts().sort_index().values
        w = torch.tensor(len(tr_df) / (2 * counts), dtype=torch.float32).to(device)
        print("class weights [normal, pneumonia]:", w.cpu().numpy().round(3))
        criterion = nn.CrossEntropyLoss(weight=w)
    else:
        criterion = nn.CrossEntropyLoss()
    # 학습 대상(requires_grad=True)인 가중치만 optimizer에 전달
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=cfg["lr"])
    # CosineAnnealingLR : 학습률을 에폭이 지날수록 코사인 곡선처럼 부드럽게 줄임
    # T_max=에폭 수 : 마지막 에폭에서 학습률이 거의 0에 가까워지도록 설정
    # cfg.get("sched") : 설정에 sched가 없으면 None → 기존 실험(E1a~E4)과 똑같이 동작
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"]) if cfg.get("sched") == "cosine" else None

    best_auc, best_state, best_ep, history = -1, None, 0, []
    for ep in range(cfg["epochs"]):
        model.train()
        # 특징 추출 모드 : 고정한 층의 BatchNorm 통계까지 바뀌지 않도록 평가 모드 유지 (fc는 영향 없음)
        if cfg["freeze"] == "fc":
            model.eval()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running += loss.item() * x.size(0)
        tr_loss = running / len(train_loader.dataset)
        # 에폭 하나가 끝날 때마다 학습률을 한 단계 낮춤 (스케줄러가 켜져 있을 때만)
        if scheduler is not None:
            scheduler.step()
        v = evaluate(model, val_loader)
        history.append({"epoch": ep + 1, "train_loss": tr_loss, "val_acc": v["acc"], "val_auc": v["auc"]})
        print(f"[{name}] Epoch {ep+1} | loss {tr_loss:.4f} | val acc {v['acc']:.4f} | val auc {v['auc']:.4f} | sens {v['sens']:.3f} | spec {v['spec']:.3f}")
        # 검증 AUC가 가장 좋은 에폭의 가중치 보관 (검증셋은 선택에만 사용, 학습에는 미사용)
        if v["auc"] > best_auc:
            best_auc, best_ep = v["auc"], ep + 1
            best_state = copy.deepcopy(model.state_dict())

    # 가장 좋았던 에폭으로 되돌린 뒤 최종 평가 · 저장 · 예측
    model.load_state_dict(best_state)
    v = evaluate(model, val_loader)
    torch.save(best_state, os.path.join(MODEL_DIR, f"{name}.pth"))

    test_probs = predict(model, test_loader)
    test_preds = (test_probs >= 0.5).astype(int)
    sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
    assert (sub["file_name"].values == test_df["file_name"].values).all()
    sub["label"] = test_preds
    sub.to_csv(os.path.join(SUB_DIR, f"{name}.csv"), index=False)

    row = {"exp": name, "desc": desc, **{k: cfg[k] for k in ["crop", "clahe", "aug", "weighted", "freeze", "lr", "epochs"]},
           "best_epoch": best_ep, "val_acc": round(v["acc"], 4), "val_auc": round(v["auc"], 4),
           "val_sens": round(v["sens"], 4), "val_spec": round(v["spec"], 4), "val_f1": round(v["f1"], 4),
           "val_fn": v["fn"], "val_fp": v["fp"],
           "test_normal_ratio": round(1 - test_preds.mean(), 3),
           "test_prob_median": round(float(np.median(test_probs)), 3),
           "minutes": round((time.time() - t0) / 60, 1)}
    # 기록표에 스케줄러 사용 여부도 남김 (없으면 "none")
    row["sched"] = cfg.get("sched", "none")
    res = log_result(row)
    print(f"\n✅ {name} done | best epoch {best_ep} | val acc {v['acc']:.4f} | val auc {v['auc']:.4f} "
          f"| test normal ratio {row['test_normal_ratio']} | test prob median {row['test_prob_median']} | {row['minutes']} min")
    return {"model": model, "val": v, "test_probs": test_probs, "history": history}

# 기본 설정 : 실험마다 여기서 바꾸고 싶은 항목만 덮어씀
BASE_CFG = {"crop": False, "clahe": False, "aug": False, "weighted": False,
            "freeze": "full", "lr": 1e-4, "epochs": 5}


# → 평가·예측·기록 함수와, 설정(cfg)만 넣으면 학습 → 최고 에폭 선택 → 저장 → 제출 파일 생성 →
#   results.csv 기록까지 한 번에 해주는 run_experiment 함수를 만들겠다

## 🧪 단계 1. E1a / E1b — 사전학습 모델 + Fine-tuning 방식 비교

- **바꾸는 것**: 모델(SimpleCNN → ResNet18), 해상도(128 → 224), 정규화(ImageNet), 에폭(2 → 5)
- **E1a**: 사전학습 층 고정, 분류층(fc)만 학습 (lr 1e-3)
- **E1b**: 전체 층 미세조정 (lr 1e-4)
- **질문**: 사전학습 모델이 Baseline보다 나은가? 어떤 Fine-tuning 방식이 X-ray에 맞는가?

In [ ]:
# E1a : ResNet18 feature extraction (fc only)


cfg_e1a = {**BASE_CFG, "freeze": "fc", "lr": 1e-3}
out_e1a = run_experiment("E1a", "ResNet18 224 freeze(fc only)", cfg_e1a)


# → 사전학습 층은 고정하고 분류층만 학습하는 특징 추출 방식으로 5 에폭 학습하겠다

[E1a] Epoch 1 | loss 0.2162 | val acc 0.9521 | val auc 0.9889 | sens 0.964 | spec 0.918
[E1a] Epoch 2 | loss 0.1215 | val acc 0.9579 | val auc 0.9920 | sens 0.990 | spec 0.866
[E1a] Epoch 3 | loss 0.1029 | val acc 0.9607 | val auc 0.9933 | sens 0.969 | spec 0.937
[E1a] Epoch 4 | loss 0.0952 | val acc 0.9655 | val auc 0.9939 | sens 0.982 | spec 0.918
[E1a] Epoch 5 | loss 0.0898 | val acc 0.9646 | val auc 0.9944 | sens 0.987 | spec 0.899

✅ E1a done | best epoch 5 | val acc 0.9646 | val auc 0.9944 | test normal ratio 0.471 | test prob median 0.592 | 1.2 min


In [ ]:
# E1b : ResNet18 full fine-tuning


cfg_e1b = {**BASE_CFG, "freeze": "full", "lr": 1e-4}
out_e1b = run_experiment("E1b", "ResNet18 224 full fine-tuning", cfg_e1b)


# → 모든 층을 작은 학습률(1e-4)로 함께 학습하는 전체 미세조정 방식으로 5 에폭 학습하겠다

[E1b] Epoch 1 | loss 0.0998 | val acc 0.9579 | val auc 0.9949 | sens 1.000 | spec 0.836
[E1b] Epoch 2 | loss 0.0305 | val acc 0.9693 | val auc 0.9974 | sens 1.000 | spec 0.881
[E1b] Epoch 3 | loss 0.0171 | val acc 0.9808 | val auc 0.9975 | sens 0.991 | spec 0.951
[E1b] Epoch 4 | loss 0.0125 | val acc 0.9895 | val auc 0.9977 | sens 0.994 | spec 0.978
[E1b] Epoch 5 | loss 0.0161 | val acc 0.9799 | val auc 0.9979 | sens 0.977 | spec 0.989

✅ E1b done | best epoch 5 | val acc 0.9799 | val auc 0.9979 | test normal ratio 0.386 | test prob median 0.982 | 1.8 min


In [ ]:
# Results table


res = pd.read_csv(RESULTS_PATH)
cols = ["exp", "desc", "best_epoch", "val_acc", "val_auc", "val_sens", "val_spec",
        "val_fn", "val_fp", "test_normal_ratio", "test_prob_median"]
# 있는 열만 골라서 보기 (E0 행에는 일부 열이 비어 있음)
display(res[[c for c in cols if c in res.columns]])


# → 지금까지의 실험 결과를 한 표로 모아 Baseline과 비교하겠다

,exp,desc,best_epoch,val_acc,val_auc,val_sens,val_spec,val_fn,val_fp,test_normal_ratio,test_prob_median
0,E0,Baseline SimpleCNN 128,NaN,0.9454,0.9832,0.9433,0.9515,44,13,0.497,0.503
1,E2,E1b + Crop,5.0,0.9875,0.9988,0.9858,0.9925,11,2,0.284,1.000
2,E2b,E1b + Crop + fixed ratio 1.3,3.0,0.9713,0.9967,0.9691,0.9776,24,6,0.296,0.998
3,E4,E1b + weighted loss,3.0,0.9684,0.9984,0.9601,0.9925,31,2,0.388,0.960
4,E1a,ResNet18 224 freeze(fc only),5.0,0.9646,0.9944,0.9871,0.8993,10,27,0.471,0.592
5,E1b,ResNet18 224 full fine-tuning,5.0,0.9799,0.9979,0.9768,0.9888,18,3,0.386,0.982


### 📊 단계 1 결과: 사전학습 모델 + Fine-tuning 방식

| 실험 | 설정 | Val Acc | Val AUC | Sens | Spec | FN | FP | 테스트 확률 중앙값 | 테스트 정상 예측 비율 | 리더보드 | 검증-리더보드 격차 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| E0 | SimpleCNN 128, 2 ep | 0.9454 | 0.9832 | 0.943 | 0.952 | 44 | 13 | 0.503 | 0.497 | 0.7468 | 19.9%p |
| E1a | ResNet18 224, fc만 학습 | 0.9646 | 0.9944 | 0.987 | 0.899 | 10 | 27 | 0.592 | 0.471 | 0.8333 | 13.1%p |
| E1b | ResNet18 224, 전체 미세조정 | **0.9799** | **0.9979** | 0.977 | **0.989** | 18 | **3** | **0.982** | **0.386** | **0.9407** | **3.9%p** |

**해석**
- **사전학습 모델의 효과**: 두 방식 모두 Baseline보다 검증 AUC 상승, 놓친 폐렴 44명 → 10~18명
- **Fine-tuning 방식 결정 → 전체 미세조정(E1b)**
  - E1a는 특이도 0.899(오진 27명), 리더보드 0.8333으로 검증 대비 13.1%p 하락 → ImageNet 특징을 그대로 쓰는 것만으로는 X-ray에 부족
  - E1b는 민감도·특이도 모두 0.97 이상, 리더보드 0.9407 → ImageNet(컬러 일반 사진)과 X-ray(흑백 의료 영상)의 차이 때문에 특징 자체를 조정해야 한다는 예상이 확인됨
- **분포 차이 완화 (핵심)**
  - 테스트 확률 중앙값 0.503 → **0.982**: 검증셋(0.979)과 비슷한 확신도 회복
  - 검증-리더보드 격차 19.9%p → 13.1%p → **3.9%p**
  - E1a는 Baseline과 큰 차이 없음 → 분포 차이를 줄인 요인은 사전학습 자체보다 **전체 미세조정**
- **남은 과제**: E1b도 624장 중 약 37장 오답, 정상 예측 비율(0.386)은 실제(약 0.375)와 비슷하나 오류가 양방향으로 상쇄된 결과 → 단계 2(Crop)의 목표
- **가설 (Grad-CAM으로 검증 예정)**: 전체 미세조정으로 더 일반적인 폐 특징을 학습하여 검은 여백 같은 단순 단서에 덜 의존하게 되었을 가능성

**주의 사항**
- E1b train loss 0.0125(4 ep) → 0.0161(5 ep) 소폭 상승, 학습 데이터를 거의 다 맞히는 수준 → 과적합 가능성 → E5에서 검토
- E0(2 epochs)와 E1(5 epochs)은 에폭 수가 달라 개선 폭에 에폭 증가 효과가 일부 포함됨

**다음 단계 기본 설정**: E1b (ResNet18, 전체 미세조정, lr 1e-4, 5 epochs)

## 🧪 단계 2. E2 — 검은 여백 Crop (S1)

- **바꾸는 것**: E1b 설정에 Crop만 추가 (나머지 동일)
- **가설 H1**: 학습 이미지의 검은 여백을 잘라내면 지름길 단서가 사라지고 테스트 이미지(여백 없음)와 형태가 비슷해져, **리더보드 성능이 오르고 검증-리더보드 격차(3.9%p)가 줄어든다**
- **확인할 것**: 검증 지표 / 테스트 확률 중앙값 / 리더보드 / 격차
- Crop은 이미지 한 장 안에서만 계산하므로 테스트 이미지에도 동일하게 적용 (테스트 통계 미사용)

In [ ]:
# Crop preview : before vs after


import matplotlib.pyplot as plt

# 확인용 샘플 : 학습 정상 2장, 학습 폐렴 2장, 테스트 2장
# EDA에서 여백이 컸던 train_3582(폐렴), 기울어진 train_5089(폐렴)를 일부러 포함
samples = [
    ("train", "train_0803.png", "NORMAL"),
    ("train", "train_0260.png", "NORMAL"),
    ("train", "train_3582.png", "PNEUMONIA"),
    ("train", "train_5089.png", "PNEUMONIA"),
    ("test",  test_df["file_name"].iloc[0], "TEST"),
    ("test",  test_df["file_name"].iloc[1], "TEST"),
]

# 2행 : 위 = 원본, 아래 = Crop 후 / 열 = 샘플 6장
fig, axes = plt.subplots(2, len(samples), figsize=(3 * len(samples), 6.5))
for col, (split, fname, tag) in enumerate(samples):
    root = TRAIN_IMG_DIR if split == "train" else TEST_IMG_DIR
    img = Image.open(os.path.join(root, fname)).convert("L")
    # 셀 4에서 정의한 crop_black_border 함수 그대로 사용
    cropped = crop_black_border(img)
    # 잘려나간 면적 비율 = 1 - (Crop 후 넓이 / 원본 넓이)
    removed = 1 - (cropped.size[0] * cropped.size[1]) / (img.size[0] * img.size[1])

    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(f"{tag}\n{fname}\n{img.size[0]}x{img.size[1]}", fontsize=9)
    axes[1, col].imshow(cropped, cmap="gray")
    axes[1, col].set_title(f"cropped {cropped.size[0]}x{cropped.size[1]}\nremoved {removed:.1%}", fontsize=9)
    for r in range(2):
        axes[r, col].axis("off")

plt.suptitle("Crop check (top: original / bottom: after crop)", fontsize=13)
plt.tight_layout()
plt.show()


# → 학습 정상·폐렴과 테스트 이미지에 Crop 함수를 적용해 원본과 나란히 보여주고,
#   잘려나간 면적 비율로 여백이 제대로 제거되는지(폐 영역은 남는지) 눈으로 확인하겠다

[X-ray 이미지 출력 제거 — 대회 데이터 재배포 금지. 해석은 04_Report/Final_Report.md 참고]


In [ ]:
# Stage 2.5 : brightness after crop (train vs test)


def mean_brightness(df, root, crop, n=None):
    # n : 샘플 개수 (None이면 전체) / random_state=SEED : 매번 같은 샘플
    files = df["file_name"] if n is None else df["file_name"].sample(n, random_state=SEED)
    vals = []
    for f in files:
        img = Image.open(os.path.join(root, f)).convert("L")
        if crop:
            img = crop_black_border(img)
        vals.append(np.array(img).mean())
    return np.array(vals)

# 학습은 600장 샘플 (02_Baseline 측정과 같은 조건), 테스트는 624장 전체
rows = []
for crop in [False, True]:
    tr = mean_brightness(tr_df, TRAIN_IMG_DIR, crop, n=600)
    te = mean_brightness(test_df, TEST_IMG_DIR, crop)
    rows.append({"crop": crop, "train_mean": round(tr.mean(), 1), "test_mean": round(te.mean(), 1),
                 "gap": round(te.mean() - tr.mean(), 1)})
bright_df = pd.DataFrame(rows)
display(bright_df)

# 차이가 얼마나 줄었는지 비율로
before, after = bright_df.loc[0, "gap"], bright_df.loc[1, "gap"]
print(f"brightness gap: {before} → {after}  ({(1 - after / before):.0%} 감소)")


# → Crop 전/후로 학습·테스트 이미지의 평균 밝기를 다시 재서,
#   밝기 차이가 검은 여백 때문이었는지(Crop 후 줄어드는지) 확인하겠다

,crop,train_mean,test_mean,gap
0,False,87.6,121.0,33.4
1,True,121.0,122.1,1.1


brightness gap: 33.4 → 1.1  (97% 감소)


In [ ]:
# E2 : E1b + Crop


# E1b 설정(전체 미세조정, lr 1e-4, 5 epochs)에 Crop만 켬
cfg_e1b = {**BASE_CFG, "freeze": "full", "lr": 1e-4}
cfg_e2 = {**cfg_e1b, "crop": True}
out_e2 = run_experiment("E2", "E1b + Crop", cfg_e2)


# → E1b와 모든 설정을 같게 두고 Crop 하나만 추가해 학습하여,
#   Crop의 효과만 분리해서 확인하겠다

[E2] Epoch 1 | loss 0.0961 | val acc 0.9875 | val auc 0.9967 | sens 0.992 | spec 0.974
[E2] Epoch 2 | loss 0.0331 | val acc 0.9799 | val auc 0.9954 | sens 0.986 | spec 0.963
[E2] Epoch 3 | loss 0.0210 | val acc 0.9761 | val auc 0.9974 | sens 0.972 | spec 0.989
[E2] Epoch 4 | loss 0.0130 | val acc 0.9875 | val auc 0.9984 | sens 0.991 | spec 0.978
[E2] Epoch 5 | loss 0.0025 | val acc 0.9875 | val auc 0.9988 | sens 0.986 | spec 0.993

✅ E2 done | best epoch 5 | val acc 0.9875 | val auc 0.9988 | test normal ratio 0.284 | test prob median 1.0 | 1.9 min


In [ ]:
# Results table (updated)


res = pd.read_csv(RESULTS_PATH)
cols = ["exp", "desc", "best_epoch", "val_acc", "val_auc", "val_sens", "val_spec",
        "val_fn", "val_fp", "test_normal_ratio", "test_prob_median"]
display(res[[c for c in cols if c in res.columns]])


# → E2까지 포함한 실험 결과를 한 표로 다시 보겠다

,exp,desc,best_epoch,val_acc,val_auc,val_sens,val_spec,val_fn,val_fp,test_normal_ratio,test_prob_median
0,E0,Baseline SimpleCNN 128,NaN,0.9454,0.9832,0.9433,0.9515,44,13,0.497,0.503
1,E2b,E1b + Crop + fixed ratio 1.3,3.0,0.9713,0.9967,0.9691,0.9776,24,6,0.296,0.998
2,E4,E1b + weighted loss,3.0,0.9684,0.9984,0.9601,0.9925,31,2,0.388,0.960
3,E1a,ResNet18 224 freeze(fc only),5.0,0.9646,0.9944,0.9871,0.8993,10,27,0.471,0.592
4,E1b,ResNet18 224 full fine-tuning,5.0,0.9799,0.9979,0.9768,0.9888,18,3,0.386,0.982
5,E2,E1b + Crop,5.0,0.9875,0.9988,0.9858,0.9925,11,2,0.284,1.000


### 📊 단계 2 결과: Crop (E2) — 생각의 흐름

**① 가설 H1**: 학습 이미지의 검은 여백을 잘라내면 지름길 단서가 사라지고 테스트 이미지와 형태가 비슷해져, 리더보드가 오르고 검증-리더보드 격차가 줄어들 것

**② Crop 동작 확인 (학습 전)**
- 여백 제거 정상 (`train_3582` 36.2% 제거), **폐 영역 잘림 없음**
- 한계: 기울어진 `train_5089`의 모서리 검은 삼각형은 남음 (줄 전체가 검은 행·열만 자르는 방식)
- 테스트 이미지도 3.7%, 7.6% 제거 → 테스트에도 약간의 여백 존재

**③ 2.5단계: Crop 후 밝기 재측정 🎯**
| | 학습 | 테스트 | 차이 |
|---|---|---|---|
| Crop 전 | 87.6 | 121.0 | 33.4 |
| Crop 후 | 121.0 | 122.1 | **1.1 (97% 감소)** |
- 밝기 차이의 97%가 검은 여백 때문 → 실제 밝기 차이는 거의 없음
- 설계 기준에 따라 **S2(밝기 대응)는 가볍게 진행**하기로 계획 조정

**④ E2 결과: 검증은 최고, 테스트 쪽 경고 ⚠️**
| 실험 | Val Acc | Val AUC | Sens | Spec | FN | FP | 테스트 확률 중앙값 | 테스트 정상 예측 비율 | 리더보드 |
|---|---|---|---|---|---|---|---|---|---|
| E1b | 0.9799 | 0.9979 | 0.977 | 0.989 | 18 | 3 | 0.982 | 0.386 | 0.9407 |
| E2 | **0.9875** | **0.9988** | 0.986 | **0.993** | **11** | **2** | 1.000 | 0.284 | **0.9022** |
- 검증 지표는 모두 개선
- 그러나 테스트 정상 예측 비율 0.284 (실제 약 0.375) → 정상을 폐렴으로 오진한 경우 최소 약 57장 → 리더보드 상한 약 0.91 → **E1b보다 하락 가능성**
- 테스트 확률 중앙값 1.000 → 테스트에서 과신
- **리더보드 결과: 0.9407 → 0.9022 (−3.85%p), 검증-리더보드 격차 3.9%p → 8.5%p** → 예측한 상한(약 0.91)과 일치 → **가설 H1 기각**, 최종 제출 선택은 E1b 유지

**⑤ 새 지름길 가설 H1'**
- Crop 결과 크기: 정상 221×187, 211×184 (정사각형에 가까움) / 폐렴 224×143, 224×144 (넓적) / 테스트 원본 1936×1554, 1612×941 (넓적)
- Resize(224×224) 시 넓적한 폐렴 이미지만 세로로 늘어남 → 모델이 **"늘어난 이미지 = 폐렴"**을 학습했을 가능성 → 넓적한 테스트 이미지가 폐렴으로 기울어짐
- 검은 여백을 없애자, 여백이 가리고 있던 **가로세로 비율 차이**가 새 지름길로 드러난 셈
- 검증: 리더보드 점수 + 가로세로 비율 진단 (아래 셀)

In [ ]:
# Diagnose : aspect ratio after crop (new shortcut?)


def aspect_ratios(df, root, crop):
    vals = []
    for f in df["file_name"]:
        img = Image.open(os.path.join(root, f)).convert("L")
        if crop:
            img = crop_black_border(img)
        # 가로 ÷ 세로 : 1이면 정사각형, 클수록 옆으로 넓적함
        vals.append(img.size[0] / img.size[1])
    return np.array(vals)

rows = []
# 학습 데이터 : 클래스별로 나눠서 측정
for label, name in [(0, "train NORMAL"), (1, "train PNEUMONIA")]:
    r = aspect_ratios(tr_df[tr_df["label"] == label], TRAIN_IMG_DIR, crop=True)
    rows.append({"group": name, "mean": r.mean(), "median": np.median(r), "p10": np.percentile(r, 10), "p90": np.percentile(r, 90)})
# 테스트 데이터 : 정답이 없으므로 전체로 측정 (관찰용, 학습·설정에는 사용하지 않음)
r = aspect_ratios(test_df, TEST_IMG_DIR, crop=True)
rows.append({"group": "test (all)", "mean": r.mean(), "median": np.median(r), "p10": np.percentile(r, 10), "p90": np.percentile(r, 90)})

display(pd.DataFrame(rows).round(2))


# → Crop 후 가로세로 비율을 정상·폐렴·테스트별로 비교해,
#   "넓적함(늘어남) = 폐렴"이라는 새로운 지름길이 생겼는지 확인하겠다

,group,mean,median,p10,p90
0,train NORMAL,1.19,1.19,1.04,1.35
1,train PNEUMONIA,1.47,1.45,1.21,1.75
2,test (all),1.46,1.44,1.17,1.75


**⑥ 가로세로 비율 진단 결과 → 가설 H1' 확인 ✅**
| 그룹 | 평균 | 중앙값 | 하위 10% | 상위 10% |
|---|---|---|---|---|
| 학습 정상 | 1.19 | 1.19 | 1.04 | 1.35 |
| 학습 폐렴 | **1.47** | **1.45** | 1.21 | 1.75 |
| 테스트 (전체) | **1.46** | **1.44** | 1.17 | 1.75 |
- Crop 후 학습 폐렴은 정상보다 확연히 넓적하고(1.47 vs 1.19), **테스트 이미지는 폐렴과 거의 같은 비율(1.46)**
- → 224×224로 맞출 때 "넓적해서 세로로 늘어난 이미지 = 폐렴"이라는 단서가 학습되고, 테스트 이미지는 정상이든 폐렴이든 모두 넓적하므로 폐렴 쪽으로 기울어짐 → E2의 정상 예측 비율 0.284, 리더보드 하락과 일치
- **근본 원인 정리**: 원본 비율 차이(정상은 정사각형에 가깝고 폐렴은 넓적) → 대회 측이 정사각형으로 맞추며 덧댄 **패딩 양의 차이** → EDA에서 본 **검은 영역 차이**와 **밝기 차이**. 즉 지금까지 본 여러 현상은 **하나의 원인(원본 가로세로 비율)**에서 나온 증상들
- **다음 실험 E2b**: 비율 정보 자체를 없애야 함
  - 비율을 유지하고 패딩하면 → 패딩 양이 다시 단서가 됨
  - 그냥 늘이면 → 늘어난 정도가 단서가 됨 (E2)
  - → **모든 이미지를 같은 비율(1.3)로 가운데를 잘라낸 뒤** 224×224로 맞춤 → 모든 이미지가 똑같이 변형되므로 비율이 단서가 될 수 없음

## 🧪 단계 2-b. E2b — Crop + 고정 비율 중앙 Crop

- **바꾸는 것**: E2(Crop)에 "모든 이미지를 가로세로 비율 1.3으로 가운데를 잘라내기"를 추가
- **가설 H1b**: 비율 정보를 없애면 E2에서 생긴 "늘어남 = 폐렴" 지름길이 사라져, **검은 여백 제거의 이점은 살리면서 리더보드 하락을 막을 수 있다**
- **비율 1.3을 고른 이유**: 학습 데이터의 정상(중앙값 1.19)과 폐렴(1.45) 사이 값 → 양쪽 모두 잘려나가는 양을 줄임 (테스트 통계는 사용하지 않음)
  - 1.3보다 넓적한 이미지 → 좌우를 잘라냄 (주로 팔·몸 바깥)
  - 1.3보다 좁은 이미지 → 위아래를 잘라냄 (주로 목·복부)
- **다른 방법을 쓰지 않은 이유**
  - 비율 유지 + 패딩: 패딩 양이 다시 단서가 됨 (원래 데이터의 문제 그대로)
  - 그냥 늘이기: 늘어난 정도가 단서가 됨 (E2의 문제)

In [ ]:
# Fixed aspect-ratio center crop & preprocess update


# TARGET_RATIO : 모든 이미지를 맞출 가로÷세로 비율 (학습 정상 1.19 ~ 폐렴 1.45 사이)
TARGET_RATIO = 1.3

def center_crop_ratio(img, ratio=TARGET_RATIO):
    """모든 이미지를 같은 가로세로 비율로 가운데 부분만 잘라냄 → 비율이 단서가 되지 않게"""
    w, h = img.size
    if w / h > ratio:
        # 기준보다 넓적함 → 가로를 잘라냄 (세로는 그대로, 가로 = 세로 × 비율)
        new_w = int(round(h * ratio))
        left = (w - new_w) // 2
        return img.crop((left, 0, left + new_w, h))
    else:
        # 기준보다 좁음 → 세로를 잘라냄 (가로는 그대로, 세로 = 가로 ÷ 비율)
        new_h = int(round(w / ratio))
        top = (h - new_h) // 2
        return img.crop((0, top, w, top + new_h))

# 셀 4의 preprocess를 덮어써서 "aspect" 옵션 추가
# cfg.get("aspect", False) : 설정에 aspect가 없으면 False로 간주 → 기존 실험 설정과도 호환
def preprocess(img, cfg):
    if cfg["crop"]:
        img = crop_black_border(img)
    if cfg.get("aspect", False):
        img = center_crop_ratio(img)
    if cfg["clahe"]:
        img = apply_clahe(img)
    return img


# → 검은 여백을 자른 뒤 모든 이미지를 가로세로 1.3 비율로 가운데만 잘라내는 함수를 만들고,
#   전처리 순서(Crop → 고정 비율 → CLAHE)에 끼워 넣겠다

In [ ]:
# Preview : crop vs crop + fixed ratio


fig, axes = plt.subplots(2, len(samples), figsize=(3 * len(samples), 6.5))
for col, (split, fname, tag) in enumerate(samples):
    root = TRAIN_IMG_DIR if split == "train" else TEST_IMG_DIR
    img = Image.open(os.path.join(root, fname)).convert("L")
    c1 = crop_black_border(img)
    c2 = center_crop_ratio(c1)
    axes[0, col].imshow(c1, cmap="gray")
    axes[0, col].set_title(f"{tag} {fname}\ncrop {c1.size[0]}x{c1.size[1]} (ratio {c1.size[0]/c1.size[1]:.2f})", fontsize=8)
    axes[1, col].imshow(c2, cmap="gray")
    axes[1, col].set_title(f"+ fixed ratio {c2.size[0]}x{c2.size[1]} (ratio {c2.size[0]/c2.size[1]:.2f})", fontsize=8)
    for r in range(2):
        axes[r, col].axis("off")
plt.suptitle("Top: crop only (E2) / Bottom: crop + fixed ratio 1.3 (E2b)", fontsize=12)
plt.tight_layout()
plt.show()


# → E2 방식과 E2b 방식의 결과를 나란히 보여주어, 모든 이미지가 같은 비율(1.30)이 되는지와
#   폐 영역이 잘려나가지 않는지 확인하겠다

[X-ray 이미지 출력 제거 — 대회 데이터 재배포 금지. 해석은 04_Report/Final_Report.md 참고]


In [ ]:
# E2b : E1b + Crop + fixed aspect ratio


cfg_e2b = {**cfg_e1b, "crop": True, "aspect": True}
out_e2b = run_experiment("E2b", "E1b + Crop + fixed ratio 1.3", cfg_e2b)


# → E2 설정에 고정 비율 중앙 Crop만 추가해 학습하여,
#   비율 지름길을 없앴을 때 테스트 성능이 회복되는지 확인하겠다

[E2b] Epoch 1 | loss 0.1252 | val acc 0.9693 | val auc 0.9954 | sens 0.991 | spec 0.907
[E2b] Epoch 2 | loss 0.0398 | val acc 0.9761 | val auc 0.9959 | sens 0.985 | spec 0.951
[E2b] Epoch 3 | loss 0.0249 | val acc 0.9713 | val auc 0.9967 | sens 0.969 | spec 0.978
[E2b] Epoch 4 | loss 0.0160 | val acc 0.9808 | val auc 0.9963 | sens 0.982 | spec 0.978
[E2b] Epoch 5 | loss 0.0280 | val acc 0.9684 | val auc 0.9961 | sens 0.966 | spec 0.974

✅ E2b done | best epoch 3 | val acc 0.9713 | val auc 0.9967 | test normal ratio 0.296 | test prob median 0.998 | 1.9 min


In [ ]:
# Results table (updated)


res = pd.read_csv(RESULTS_PATH)
cols = ["exp", "desc", "best_epoch", "val_acc", "val_auc", "val_sens", "val_spec",
        "val_fn", "val_fp", "test_normal_ratio", "test_prob_median"]
display(res[[c for c in cols if c in res.columns]])


# → E2b까지 포함한 실험 결과를 한 표로 다시 보겠다

,exp,desc,best_epoch,val_acc,val_auc,val_sens,val_spec,val_fn,val_fp,test_normal_ratio,test_prob_median
0,E0,Baseline SimpleCNN 128,NaN,0.9454,0.9832,0.9433,0.9515,44,13,0.497,0.503
1,E4,E1b + weighted loss,3.0,0.9684,0.9984,0.9601,0.9925,31,2,0.388,0.960
2,E1a,ResNet18 224 freeze(fc only),5.0,0.9646,0.9944,0.9871,0.8993,10,27,0.471,0.592
3,E1b,ResNet18 224 full fine-tuning,5.0,0.9799,0.9979,0.9768,0.9888,18,3,0.386,0.982
4,E2,E1b + Crop,5.0,0.9875,0.9988,0.9858,0.9925,11,2,0.284,1.000
5,E2b,E1b + Crop + fixed ratio 1.3,3.0,0.9713,0.9967,0.9691,0.9776,24,6,0.296,0.998


In [ ]:
# E4 : E1b + class-weighted loss


# E1b 설정을 여기서 다시 정의 → 위쪽 실험 셀(E1a~E2b)을 다시 돌리지 않아도 이 셀만으로 실행 가능
cfg_e1b = {**BASE_CFG, "freeze": "full", "lr": 1e-4}
# E1b 설정에 weighted만 켬 (Crop·aspect 없음)
cfg_e4 = {**cfg_e1b, "weighted": True}
out_e4 = run_experiment("E4", "E1b + weighted loss", cfg_e4)


# → E1b와 같은 설정에서 손실함수에 클래스 가중치만 추가해 학습하여,
#   불균형 보정이 정상·폐렴 판단 균형과 테스트 성능에 주는 효과만 분리해서 확인하겠다

class weights [normal, pneumonia]: [1.944 0.673]
[E4] Epoch 1 | loss 0.1117 | val acc 0.9789 | val auc 0.9961 | sens 1.000 | spec 0.918
[E4] Epoch 2 | loss 0.0455 | val acc 0.9866 | val auc 0.9969 | sens 0.988 | spec 0.981
[E4] Epoch 3 | loss 0.0212 | val acc 0.9684 | val auc 0.9984 | sens 0.960 | spec 0.993
[E4] Epoch 4 | loss 0.0101 | val acc 0.9895 | val auc 0.9980 | sens 0.994 | spec 0.978
[E4] Epoch 5 | loss 0.0111 | val acc 0.9856 | val auc 0.9984 | sens 0.988 | spec 0.978

✅ E4 done | best epoch 3 | val acc 0.9684 | val auc 0.9984 | test normal ratio 0.388 | test prob median 0.96 | 1.8 min


In [ ]:
# Results table (updated)


res = pd.read_csv(RESULTS_PATH)
cols = ["exp", "desc", "best_epoch", "val_acc", "val_auc", "val_sens", "val_spec",
        "val_fn", "val_fp", "test_normal_ratio", "test_prob_median"]
display(res[[c for c in cols if c in res.columns]])


# → E4까지 포함한 실험 결과를 한 표로 다시 보겠다

,exp,desc,best_epoch,val_acc,val_auc,val_sens,val_spec,val_fn,val_fp,test_normal_ratio,test_prob_median
0,E0,Baseline SimpleCNN 128,NaN,0.9454,0.9832,0.9433,0.9515,44,13,0.497,0.503
1,E1a,ResNet18 224 freeze(fc only),5.0,0.9646,0.9944,0.9871,0.8993,10,27,0.471,0.592
2,E1b,ResNet18 224 full fine-tuning,5.0,0.9799,0.9979,0.9768,0.9888,18,3,0.386,0.982
3,E2,E1b + Crop,5.0,0.9875,0.9988,0.9858,0.9925,11,2,0.284,1.000
4,E2b,E1b + Crop + fixed ratio 1.3,3.0,0.9713,0.9967,0.9691,0.9776,24,6,0.296,0.998
5,E4,E1b + weighted loss,3.0,0.9684,0.9984,0.9601,0.9925,31,2,0.388,0.960


### 📊 단계 4 결과: 클래스 가중치 손실 (E4) — 생각의 흐름

**가설**: 정상(소수 클래스)을 틀렸을 때 벌점을 키우면(가중치 정상 1.944 / 폐렴 0.673) 정상 판단이 좋아져 특이도와 리더보드가 오를 것

| 실험 | Val AUC | Sens | Spec | FN | FP | 테스트 정상 예측 비율 | 테스트 확률 중앙값 | 최고 에폭 | 리더보드 |
|---|---|---|---|---|---|---|---|---|---|
| E1b | 0.9979 | 0.977 | 0.989 | 18 | 3 | 0.386 | 0.982 | 5 | **0.9407** |
| E4 | 0.9984 | 0.960 | 0.993 | 31 | 2 | 0.388 | 0.960 | 3 | 0.9327 |

**① 예상대로 된 것**: FP 3 → 2, 특이도 0.989 → 0.993 (정상을 더 신경 씀)

**② 예상 못 한 부작용**: FN 18 → 31 — 정상 쪽으로 기울면서 애매한 폐렴을 정상으로 판정 → 의료에서 더 위험한 실수가 늘어남

**③ 숨어 있던 문제: 최고 에폭 선택 기준**

| Epoch | Val AUC | Sens | Spec |
|---|---|---|---|
| 3 (선택됨) | 0.9984 | 0.960 | 0.993 |
| 4 | 0.9980 | 0.994 | 0.978 |
| 5 | 0.9984 | 0.988 | 0.978 |

- 3·5 에폭의 AUC가 소수점 넷째 자리까지 같았지만, "더 클 때만 교체(`>`)" 규칙 때문에 먼저 나온 3 에폭이 선택됨
- AUC는 임계값과 무관한 지표라, AUC가 같아도 0.5 기준의 실수 종류(FN·FP)는 크게 다를 수 있음

**④ 테스트**: 정상 예측 비율은 0.388로 E1b와 거의 같지만, 제출 파일을 비교하니 **624장 중 25장의 예측이 바뀜** (정상→폐렴 12장, 폐렴→정상 13장)
- 리더보드 0.9407(587장 정답) → 0.9327(582장 정답): 바뀐 25장 중 **E1b가 15장, E4가 10장** 맞힘
- 비율이 같아도 오류가 서로 상쇄될 수 있어, 간접 지표만으로는 결과를 예측할 수 없었음

**결론**: 가중치 손실은 **미채택**, 기준 모델은 E1b 유지. 단, 에폭 선택 기준이 결과에 영향을 줬을 가능성을 한계로 기록

## 🧪 단계 5. E5 — 학습률 스케줄러 (S4)

**가설**: E1b의 train loss가 4 → 5 에폭에서 0.0125 → 0.0161로 **오히려 올라감** → 학습 막바지에도 학습률이 처음(1e-4)과 같아서 최적점 근처에서 흔들렸을 가능성.
학습률을 점점 줄이면(코사인 스케줄러) 막바지 학습이 안정되어 검증·테스트 성능이 좋아질 것

| 에폭 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| E1b (고정) | 1.0e-4 | 1.0e-4 | 1.0e-4 | 1.0e-4 | 1.0e-4 |
| E5 (코사인) | 1.0e-4 | 0.90e-4 | 0.65e-4 | 0.35e-4 | 0.10e-4 |

- 바꾸는 것은 스케줄러 **하나뿐**, 나머지는 E1b와 동일 → 효과 분리
- 셀 6 `run_experiment`에 `cfg.get("sched")` 옵션 추가 (없으면 기존과 똑같이 동작 → 이전 실험 재현성 유지)

In [ ]:
# E5 : E1b + cosine learning-rate scheduler (S4)


# E1b 설정 다시 정의 (위쪽 실험 셀을 다시 돌리지 않아도 이 셀만으로 실행 가능)
cfg_e1b = {**BASE_CFG, "freeze": "full", "lr": 1e-4}

# sched="cosine" : 학습률을 1e-4에서 시작해 5 에폭 동안 점점 줄임
# 나머지 설정은 E1b와 완전히 같음 → 스케줄러의 효과만 분리
cfg_e5 = {**cfg_e1b, "sched": "cosine"}

out_e5 = run_experiment("E5", "E1b + cosine LR scheduler", cfg_e5)


# → E1b와 같은 설정에서 학습률 스케줄러만 추가해 학습하여,
#   막바지 학습이 안정되면서 검증·테스트 성능이 좋아지는지 확인하겠다

[E5] Epoch 1 | loss 0.0998 | val acc 0.9579 | val auc 0.9949 | sens 1.000 | spec 0.836
[E5] Epoch 2 | loss 0.0290 | val acc 0.9847 | val auc 0.9977 | sens 1.000 | spec 0.940
[E5] Epoch 3 | loss 0.0125 | val acc 0.9856 | val auc 0.9977 | sens 0.988 | spec 0.978
[E5] Epoch 4 | loss 0.0039 | val acc 0.9885 | val auc 0.9983 | sens 0.988 | spec 0.989
[E5] Epoch 5 | loss 0.0038 | val acc 0.9923 | val auc 0.9983 | sens 0.996 | spec 0.981

✅ E5 done | best epoch 4 | val acc 0.9885 | val auc 0.9983 | test normal ratio 0.341 | test prob median 0.989 | 1.8 min


In [ ]:
# Results table (updated)


res = pd.read_csv(RESULTS_PATH)
cols = ["exp", "desc", "best_epoch", "val_acc", "val_auc", "val_sens", "val_spec",
        "val_fn", "val_fp", "test_normal_ratio", "test_prob_median"]
display(res[[c for c in cols if c in res.columns]])


# → E5까지 포함한 실험 결과를 한 표로 다시 보겠다

,exp,desc,best_epoch,val_acc,val_auc,val_sens,val_spec,val_fn,val_fp,test_normal_ratio,test_prob_median
0,E0,Baseline SimpleCNN 128,NaN,0.9454,0.9832,0.9433,0.9515,44,13,0.497,0.503
1,E1a,ResNet18 224 freeze(fc only),5.0,0.9646,0.9944,0.9871,0.8993,10,27,0.471,0.592
2,E1b,ResNet18 224 full fine-tuning,5.0,0.9799,0.9979,0.9768,0.9888,18,3,0.386,0.982
3,E2,E1b + Crop,5.0,0.9875,0.9988,0.9858,0.9925,11,2,0.284,1.000
4,E2b,E1b + Crop + fixed ratio 1.3,3.0,0.9713,0.9967,0.9691,0.9776,24,6,0.296,0.998
5,E4,E1b + weighted loss,3.0,0.9684,0.9984,0.9601,0.9925,31,2,0.388,0.960
6,E5,E1b + cosine LR scheduler,4.0,0.9885,0.9983,0.9884,0.9888,9,3,0.341,0.989
